# Healthcare NLP on Azure ML Managed Online Endpoints

This notebook is the executable companion to `readme_deployment_azureml_endpoint.md`. Use the README for the deployment narrative, prerequisites, and troubleshooting guidance. Use this notebook to run the Azure ML deployment workflow step by step.

**Workflow covered here:**
1. Install local notebook dependencies.
2. Validate Spark NLP + Spark NLP for Healthcare.
3. Load and test the medical NER pipeline.
4. Save and register the pipeline as an Azure ML model.
5. Create a Managed Online Endpoint and deployment.
6. Route endpoint traffic and collect logs for troubleshooting.

> Security note: keep `license_keys.json` local/private. Do not commit real AWS keys or Healthcare NLP license tokens. For production, prefer Azure Key Vault as described in the README.


## 1. Install notebook dependencies

These cells install Spark NLP, Healthcare NLP, PySpark, and Azure ML SDK dependencies needed by the notebook runtime. Match these versions with the Docker image and endpoint deployment described in the README.


In [ ]:
import sys
!{sys.executable} -m pip install spark-nlp pyspark==3.5.0

In [ ]:
!{sys.executable} -m pip install spark-nlp-jsl==6.1.0 --extra-index-url https://pypi.johnsnowlabs.com/<YOUR_LICENSE_TOKEN> --upgrade

In [ ]:
!{sys.executable} -m pip install -U "azure-ai-ml>=1.19.0" "azure-identity>=1.16.0"

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

print("✅ Azure ML SDK working with Marshmallow", __import__("marshmallow").__version__)

## 2. Configure Healthcare NLP license and validate Spark

The notebook expects a local `license_keys.json` file containing placeholders such as `SPARK_NLP_LICENSE`, `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, and `ACR_NAME`. This keeps the deployment code reproducible while avoiding hardcoded secrets in notebook cells.


In [ ]:
import os
import json

import sparknlp_jsl

with open("license_keys.json") as f:
    license_keys = json.load(f)

os.environ.update(license_keys)
spark = sparknlp_jsl.start(secret=license_keys["SPARK_NLP_LICENSE"])
spark

In [ ]:
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import SentenceDetectorDLModel
from pyspark.ml import Pipeline

data = spark.createDataFrame([
    [1, "Azure Machine Learning is powerful. Spark NLP runs perfectly here!"]
]).toDF("id", "text")

data.show(truncate=False)

## 3. Load and test the medical NER pipeline

This section loads the `ner_jsl_pipeline` pretrained Healthcare pipeline and runs a local inference smoke test before deployment.

In [ ]:
from sparknlp.pretrained import PretrainedPipeline

medical_ner_pipeline = PretrainedPipeline("ner_jsl_pipeline", "en", "clinical/models")

In [ ]:
text = '''The patient is a 21-day-old Caucasian male here for 2 days of congestion - mom has been suctioning yellow discharge from the patient's nares, plus she has noticed some mild problems with his breathing while feeding (but negative for any perioral cyanosis or retractions). Additionally, there is no side effect observed after Influenza vaccine. One day ago, mom also noticed a tactile temperature and gave the patient Tylenol. Baby also has had some decreased p.o. intake. His normal breast-feeding is down from 20 minutes q.2h. to 5 to 10 minutes secondary to his respiratory congestion. He sleeps well, but has been more tired and has been fussy over the past 2 days. The parents noticed no improvement with albuterol treatments given in the ER. His urine output has also decreased; normally he has 8 to 10 wet and 5 dirty diapers per 24 hours, now he has down to 4 wet diapers per 24 hours. Mom denies any diarrhea. His bowel movements are yellow colored and soft in nature.'''

In [ ]:
result = medical_ner_pipeline.annotate(text)
result['ner_chunk']

## 4. Save the pipeline model locally

Azure ML needs a model artifact. This section saves the Spark NLP pipeline locally so it can be copied into the workspace and registered.

In [ ]:
import os
save_path = os.path.expanduser("~/spark_nlp_jsl_medical_ner_pipeline")
save_path

In [ ]:
medical_ner_pipeline.model.write().overwrite().save(save_path)

## 5. Register the model in Azure ML

This registers the saved Spark NLP Healthcare pipeline as an Azure ML model asset that can be attached to a Managed Online Deployment.

In [ ]:
import shutil, os

src = os.path.expanduser("~/spark_nlp_jsl_medical_ner_pipeline")
dst = "./spark_nlp_jsl_medical_ner_pipeline"

if os.path.exists(dst):
    shutil.rmtree(dst)

shutil.copytree(src, dst)
print("✅ Model copied locally to:", os.path.abspath(dst))

In [ ]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Model

ml_client = MLClient.from_config(DefaultAzureCredential())

registered_model = ml_client.models.create_or_update(
    Model(
        name="spark_nlp_jsl_medical_ner_pipeline",
        path="./spark_nlp_jsl_medical_ner_pipeline",
        description="Medical NER pipeline with Spark NLP"
    )
)

## 6. Create the endpoint and deployment

This section creates the Managed Online Endpoint, attaches the registered model, uses the custom ACR image, and configures the `score-jsl.py` scoring script.

In [ ]:
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint, ManagedOnlineDeployment,
    Model, Environment, CodeConfiguration, ProbeSettings
)
from azure.identity import DefaultAzureCredential

endpoint_name = "jsl-medical-ner-endpoint"

# 1️⃣ Endpoint
endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    auth_mode="key"
)
ml_client.begin_create_or_update(endpoint).wait()

In [ ]:
SPARK_NLP_LICENSE = license_keys['SPARK_NLP_LICENSE']
AWS_ACCESS_KEY_ID = license_keys['AWS_ACCESS_KEY_ID']
AWS_SECRET_ACCESS_KEY = license_keys['AWS_SECRET_ACCESS_KEY']

In [ ]:
# 2️⃣ Environment
ACR_NAME = license_keys['ACR_NAME']
env = Environment(
    name="spark-nlp-env",
    image=f"{ACR_NAME}.azurecr.io/spark-nlp-jsl:6.1.0"
)

model = registered_model.id
environment= env

# 3️⃣ Deployment
deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=endpoint_name,
    model=model,
    environment=environment,
    instance_type="Standard_F4s_v2",
    instance_count=1,
    code_configuration=CodeConfiguration(
        code="./",
        scoring_script="score-jsl.py"
    ),
    liveness_probe=ProbeSettings(
        initial_delay=900,  # wait 15 min before first health check
        period=60           # check every 60 seconds
    ),
    readiness_probe=ProbeSettings(
        initial_delay=900,
        period=60
    ),
    environment_variables={
        "GUNICORN_CMD_ARGS": "--timeout 1200",  # increase from 300s to 20min
        "AWS_ACCESS_KEY_ID": AWS_ACCESS_KEY_ID,
        "AWS_SECRET_ACCESS_KEY": AWS_SECRET_ACCESS_KEY,
        "SPARK_NLP_LICENSE": SPARK_NLP_LICENSE,
        "SECRET": "true"
    }
)
ml_client.begin_create_or_update(deployment).wait()


## 💻 Supported Azure ML VM Instance Types
As you may have seen on the code above we are using **Standard_F4s_v2** as instance type

For the most up-to-date list of supported instance types for **Managed Online Endpoints**, visit the official Azure documentation:  
🔗 [Supported VM SKUs for Managed Online Endpoints](https://learn.microsoft.com/en-us/azure/machine-learning/reference-managed-online-endpoints-vm-sku-list)

| **SKU Family** | **Example VM Size** | **vCPUs** | **RAM (GB)** | **GPU** | **Recommended Use Case** |
|----------------|--------------------|------------|---------------|----------|--------------------------|
| **General Purpose (CPU)** | Standard_F2s_v2 | 2 | 4 | ❌ | Small test deployments, debugging |
|  | Standard_F4s_v2 | 4 | 8 | ❌ | Lightweight NLP inference |
|  | Standard_D4_v2 | 4 | 14 | ❌ | Mid-size model serving |
| **Compute Optimized** | Standard_F8s_v2 | 8 | 16 | ❌ | High-throughput CPU inference |
| **Memory Optimized** | Standard_E4s_v3 | 4 | 32 | ❌ | Models with large embeddings |
|  | Standard_E8s_v3 | 8 | 64 | ❌ | Multi-document or batch NLP workloads |
| **GPU Accelerated** | Standard_NC6 | 6 | 56 | ✅ | GPU inference for deep NLP models |
|  | Standard_NC12 | 12 | 112 | ✅ | High-performance distributed inference |
| **Low Cost / Testing** | Standard_B2ms | 2 | 8 | ❌ | Development & experimentation only |

⚠️ *Note:* GPU SKUs require quota approval and may not be available in all regions.  
For production NER pipelines, **Standard_F4s_v2** or **Standard_E4s_v3** are recommended for a good balance of performance and cost.


In [ ]:
# Delete use just in case something went wrong when creating the endpoint
#from azure.ai.ml import MLClient
#from azure.identity import DefaultAzureCredential

#endpoint_name = "jsl-medical-ner-endpoint"
#ml_client = MLClient.from_config(DefaultAzureCredential())
#ml_client.online_endpoints.begin_delete(name=endpoint_name).wait()

## 7. Get endpoint details

After deployment, retrieve the scoring URI and provisioning state. Use this URI with the test request shown in the README.

In [ ]:
endpoint = ml_client.online_endpoints.get(endpoint_name)
print("Scoring URI:", endpoint.scoring_uri)

In [ ]:
deployments = ml_client.online_deployments.list(endpoint_name=endpoint_name)
for d in deployments:
    print(f"Deployment: {d.name}, Status: {d.provisioning_state}")

## 8. Route endpoint traffic

Managed Online Endpoints support blue/green routing. Start with 100% traffic to the validated deployment, then adjust for rollout or rollback.

When deploying a Managed Online Endpoint, Azure ML allows **traffic routing** between multiple deployments (for example, “blue” and “green”).  
This enables **safe rollouts**, **A/B testing**, and **zero-downtime updates**.

### 🔧 Configuration Overview

Traffic configuration determines how much of the endpoint’s incoming requests are routed to each deployment.  
It’s defined during or after deployment like this:

```python
endpoint.traffic = {"blue": 100}
```

Or to gradually roll out a new version (e.g., “green”):

```python
endpoint.traffic = {"blue": 90, "green": 10}
```

This means:
- 90% of incoming requests go to the stable “blue” deployment.
- 10% go to the new “green” deployment for validation.

Once the new version is verified, you can safely shift traffic to 100%:

```python
endpoint.traffic = {"green": 100}
```

In [ ]:
from azure.ai.ml.entities import ManagedOnlineEndpoint

# Get existing endpoint
endpoint = ml_client.online_endpoints.get(endpoint_name)

# Route 100% traffic to the "blue" deployment
endpoint.traffic = {"blue": 100}

ml_client.online_endpoints.begin_create_or_update(endpoint).wait()

## 9. Collect deployment logs for troubleshooting

Use this when the endpoint fails to provision, the container does not start, or the first scoring request times out.

You can save the logs in case you have some error during deployment to troubleshoot it

In [ ]:
from datetime import datetime

# Set output path with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = f"deployment_logs_{timestamp}.txt"

# Fetch logs (increase lines if needed)
logs = ml_client.online_deployments.get_logs(
    name="blue",
    endpoint_name=endpoint_name,
    lines=2000  # you can raise to 2000 if the logs are long
)

# Save to file
with open(log_file, "w") as f:
    f.write(logs)

print(f"✅ Logs saved to {log_file}")

In [ ]:
!ls -lh